In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
import cv2
import os
import re

In [ ]:
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_590'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_846'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_1158'
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ncell_192_64_64_dts_3.659e-07_ndir_2348_stored'
R = 1170000.0 # cm
r_sphere_radius = R # cm
r_sphere_center_cm = ( 1425937.5, 1425937.5, 1425937.5 ) # cm
r_sphere_radius_index = ( 32.0, 32.0, 32.0 ) # cells
r_shadowing_sphere_radius = 1.5*R/4.5 # cm

r_shadowing_sphere_center_cm = ( 6120562.5, 1425937.5, 1425937.5 ) # cm
r_shadowing_sphere_radius_index = ( 139.0, 32.0, 32.0 ) # cells

r_shadowing_sphere_center_cm = ( 4935937.5, 1425937.5, 1425937.5 )
r_shadowing_sphere_radius_index = ( 112.0, 32.0, 32.0 )

mesh_files = glob.glob(os.path.join(data_directory, 'mesh_plt*.h5'))
mesh_files_nums = [filenames.split('mesh_plt')[1].split('.h5')[0] for filenames in mesh_files]
mesh_files_sorted = [f for _, f in sorted(zip(mesh_files_nums, mesh_files))]
mesh_files_sorted = np.array(mesh_files_sorted)
nfiles = len(mesh_files_sorted)
print(f'Found {len(mesh_files_sorted)} mesh files.')

In [ ]:
rho_ye_T_h5py = h5py.File(data_directory+'/rho_Ye_T.hdf5', 'r')

# number of cells
Nx = np.array(rho_ye_T_h5py['/ncellsx'])
Ny = np.array(rho_ye_T_h5py['/ncellsy'])
Nz = np.array(rho_ye_T_h5py['/ncellsz'])
print(f'Nx = {Nx}, Ny = {Ny}, Nz = {Nz}')

# cell size
dx = ( np.array(rho_ye_T_h5py['/xmax']) - np.array(rho_ye_T_h5py['/xmin']) ) / np.array(rho_ye_T_h5py['/ncellsx']) # cm
dy = ( np.array(rho_ye_T_h5py['/ymax']) - np.array(rho_ye_T_h5py['/ymin']) ) / np.array(rho_ye_T_h5py['/ncellsy']) # cm
dz = ( np.array(rho_ye_T_h5py['/zmax']) - np.array(rho_ye_T_h5py['/zmin']) ) / np.array(rho_ye_T_h5py['/ncellsz']) # cm
print(f'dx = {dx} cm, dy = {dy} cm, dz = {dz} cm')

# cell centers
xc = np.linspace(dx / 2, dx * (Nx - 0.5), Nx) # cm
yc = np.linspace(dy / 2, dy * (Ny - 0.5), Ny) # cm
zc = np.linspace(dz / 2, dz * (Nz - 0.5), Nz) # cm

# cell centers mesh
Xc, Yc, Zc = np.meshgrid(xc, yc, zc, indexing='ij') # cm

bh_radius = 5.43e+05 # cm
bh_center_x = 48.0e+5 # cm
bh_center_y = 48.0e+5 # cm
bh_center_z = 16.0e+5 # cm

distance_from_bh = np.sqrt( (Xc - bh_center_x)**2 + (Yc - bh_center_y)**2 + (Zc - bh_center_z)**2 )
# mask_bh = (distance_from_bh < bh_radius + 1*np.max([dx,dy,dz]))
mask_bh = (distance_from_bh < bh_radius)
mask_x = (Xc < dx) | (Xc > (Nx - 1) * dx)
mask_y = (Yc < dy) | (Yc > (Ny - 1) * dy)
mask_z = (Zc < dz) | (Zc > (Nz - 1) * dz)
mask = mask_bh | mask_x | mask_y | mask_z

rho_ye_T_h5py.close()

In [ ]:
import sys
from pathlib import Path

_plot_functions_dir = Path(
    "/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/plot_functions"
)
if str(_plot_functions_dir) not in sys.path:
    sys.path.insert(0, str(_plot_functions_dir))

import plot_functions as pf


In [ ]:
x_slice_indx = 96 # cell number
y_slice_indx = 32 # cell number
z_slice_indx = 32 # cell number
time_indx    = -1 # last time step

# for time_indx in [8,12,16,20,24,28,32,36,40]:
# for time_indx in [1,2,3,4,5,6,7,8,9,10]:

for filepath in [mesh_files_sorted[-1]]:
# for filepath in mesh_files_sorted:

    print(f'Processing file: {filepath}')

    alldata_h5 = h5py.File(filepath, 'r')
    alldata_dict = {key: np.array(alldata_h5[key]) for key in alldata_h5.keys()}
    alldata_h5.close()
    alldata = alldata_dict.copy()
    print("Keys in alldata_dict:", list(alldata.keys()))

    # if time_indx==0:
    if True:
        doshow = True
    else:
        doshow = False

    time_ms = np.array(alldata['t(s)']) * 1.0e3 # ms

    # global min: -13.821983599860665 global max: 24.104102117542855
    max_cbar = 24.104102117542855
    min_cbar = 24.104102117542855 - 10
    # min_cbar = 0
    # max_cbar = 2e22
    # min_cbar = 1e15
    # max_cbar = None
    # min_cbar = None

    ###############################################
    # Neutrino diagonal terms
    ###############################################

    # # Electron neutrino number density
    zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, 1:-1, z_slice_indx])
    zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, y_slice_indx, 1:-1])
    zup = np.log10(zup)
    zlow = np.log10(zlow)
    # zup = zup
    # zlow = zlow
    print("global min:", np.nanmin([np.nanmin(zup), np.nanmin(zlow)]), "global max:", np.nanmax([np.nanmax(zup), np.nanmax(zlow)]))
    colorbarlabel = fr'$\log ( \, n_{{\nu_{{e}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    filename=f'./plots/{filepath.split('mesh_plt')[1].split('.h5')[0]}_nee'
    mincbar=min_cbar
    maxcbar=max_cbar
    # mincbar=None
    # maxcbar=None
    
    pf.plot_color_map(
        x=Xc[1:-1, 1:-1, z_slice_indx] / 1e5,
        y=Yc[1:-1, 1:-1, z_slice_indx] / 1e5,
        z=zup,
        bh_r=6.8,
        bh_x=48.0,
        bh_y=48.0,
        min_cb=mincbar,
        max_cb=maxcbar,
        x_label=r'$x \, (\mathrm{km})$',
        y_label=r'$y \, (\mathrm{km})$',
        title=fr'$z={Zc[0, 0, z_slice_indx] / 1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$',
        cbar_label=colorbarlabel,
        colormap='rainbow',
        filename=f'{filename}_xy.png',
        r_sphere_center_cm=r_sphere_center_cm,
        r_sphere_radius=r_sphere_radius,
        r_shadowing_sphere_center_cm=r_shadowing_sphere_center_cm,
        r_shadowing_sphere_radius=r_shadowing_sphere_radius,
        doshow=doshow,
        dosave=True,
    )
    pf.plot_color_map(
        x=Xc[1:-1, y_slice_indx, 1:-1] / 1e5,
        y=Zc[1:-1, y_slice_indx, 1:-1] / 1e5,
        z=zlow,
        bh_r=6.8,
        bh_x=48.0,
        bh_y=16.0,
        min_cb=mincbar,
        max_cb=maxcbar,
        x_label=r'$x \, (\mathrm{km})$',
        y_label=r'$z \, (\mathrm{km})$',
        title=fr'$y={Yc[0, y_slice_indx, 0] / 1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$',
        cbar_label=colorbarlabel,
        colormap='rainbow',
        filename=f'{filename}_xz.png',
        r_sphere_center_cm=r_sphere_center_cm,
        r_sphere_radius=r_sphere_radius,
        r_shadowing_sphere_center_cm=r_shadowing_sphere_center_cm,
        r_shadowing_sphere_radius=r_shadowing_sphere_radius,
        doshow=doshow,
        dosave=True,
    )

### Analytic number-density slices

Read `shadow_casting_analytic_nee_slices_ncell_192_64_64.h5` and plot the analytic electron-neutrino number density. The file holds the number density `n` and flux `F` on the `z = 32` (`xy`) and `y = 32` (`xz`) slices of the `192 x 64 x 64` mesh. The maps show `log10(n)`.

In [ ]:
analytic_slice_path = (
    Path(data_directory).parent
    / "shadow_casting_analytic_nee_slices_ncell_192_64_64.h5"
)
with h5py.File(analytic_slice_path, "r") as hf:
    x_cm = np.array(hf["x_cm"])
    y_cm = np.array(hf["y_cm"])
    z_cm = np.array(hf["z_cm"])
    n_xy = np.array(hf["xy/n"])
    n_xz = np.array(hf["xz/n"])
    z_slice_cm = float(hf["xy"].attrs["z_cm"])
    y_slice_cm = float(hf["xz"].attrs["y_cm"])

def log10_density(number_density):
    logged = np.full(number_density.shape, np.nan, dtype=np.float64)
    positive = number_density > 0.0
    logged[positive] = np.log10(number_density[positive])
    return logged

n_xy_log = log10_density(n_xy)
n_xz_log = log10_density(n_xz)
xx_xy, yy_xy = np.meshgrid(x_cm / 1e5, y_cm / 1e5, indexing="ij")
xx_xz, zz_xz = np.meshgrid(x_cm / 1e5, z_cm / 1e5, indexing="ij")

max_cbar = 24.104102117542855
min_cbar = max_cbar - 10
colorbarlabel = r"$\log ( \, n_{\nu_{e}}\,[\mathrm{cm}^{-3}]\,)$"
analytic_plot = Path("./plots") / "shadow_casting_analytic_nee_slices_ncell_192_64_64"
analytic_plot.parent.mkdir(exist_ok=True)

print(f"slice file: {analytic_slice_path}")
print(f"xy log10 n min/max: {np.nanmin(n_xy_log):.6g} {np.nanmax(n_xy_log):.6g}")
print(f"xz log10 n min/max: {np.nanmin(n_xz_log):.6g} {np.nanmax(n_xz_log):.6g}")

pf.plot_color_map(
    x=xx_xy[1:-1, 1:-1],
    y=yy_xy[1:-1, 1:-1],
    z=n_xy_log[1:-1, 1:-1],
    bh_r=6.8,
    bh_x=48.0,
    bh_y=48.0,
    min_cb=min_cbar,
    max_cb=max_cbar,
    x_label=r"$x \, (\mathrm{km})$",
    y_label=r"$y \, (\mathrm{km})$",
    title=fr"$z={z_slice_cm / 1e5:.1f}\,\mathrm{{km}}$",
    cbar_label=colorbarlabel,
    colormap="rainbow",
    filename=f"{analytic_plot}_xy.png",
    r_sphere_center_cm=r_sphere_center_cm,
    r_sphere_radius=r_sphere_radius,
    r_shadowing_sphere_center_cm=r_shadowing_sphere_center_cm,
    r_shadowing_sphere_radius=r_shadowing_sphere_radius,
    doshow=True,
    dosave=True,
)
pf.plot_color_map(
    x=xx_xz[1:-1, 1:-1],
    y=zz_xz[1:-1, 1:-1],
    z=n_xz_log[1:-1, 1:-1],
    bh_r=6.8,
    bh_x=48.0,
    bh_y=16.0,
    min_cb=min_cbar,
    max_cb=max_cbar,
    x_label=r"$x \, (\mathrm{km})$",
    y_label=r"$z \, (\mathrm{km})$",
    title=fr"$y={y_slice_cm / 1e5:.1f}\,\mathrm{{km}}$",
    cbar_label=colorbarlabel,
    colormap="rainbow",
    filename=f"{analytic_plot}_xz.png",
    r_sphere_center_cm=r_sphere_center_cm,
    r_sphere_radius=r_sphere_radius,
    r_shadowing_sphere_center_cm=r_shadowing_sphere_center_cm,
    r_shadowing_sphere_radius=r_shadowing_sphere_radius,
    doshow=True,
    dosave=True,
)


### Cell selection

Single-cell analysis of the theoretical and numerical distribution functions. The next cells use one mesh cell, `(cell_indx, cell_indy, cell_indz)`.


In [ ]:
cell_indx = 50
cell_indy = 32
cell_indz = 32

### Numerical distribution function

Single-cell analysis of the theoretical and numerical distribution functions. This cell reads the particles in the chosen cell. `N00_Re` is the electron-neutrino number content, and the numerical occupation is `f_ee = N00_Re * (2*pi*hbar*c)^3 / Vphase`. Particle positions and momenta are kept for the theoretical comparison.


In [ ]:
import sys
from pathlib import Path

_phys_const_dir = Path(
    "/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/phys_const"
)
if str(_phys_const_dir) not in sys.path:
    sys.path.insert(0, str(_phys_const_dir))
from phys_const import PhysConst

# N00_Re is the electron-neutrino number content of each macroparticle.
# f = N00_Re * (2*pi*hbar*c)^3 / Vphase. Plotfile N and Vphase share the
# same cell-volume factor, so this f is the occupation number.
cell_path = (
    Path(data_directory)
    / "plt00768_particles"
    / f"cell_{cell_indx}_{cell_indy}_{cell_indz}.h5"
)
with h5py.File(cell_path, "r") as hf:
    pos_x = np.array(hf["pos_x"])
    pos_y = np.array(hf["pos_y"])
    pos_z = np.array(hf["pos_z"])
    pupx = np.array(hf["pupx"])
    pupy = np.array(hf["pupy"])
    pupz = np.array(hf["pupz"])
    pupt = np.array(hf["pupt"])
    N00_Re = np.array(hf["N00_Re"])
    Vphase = np.array(hf["Vphase"])

f_ee = PhysConst.hc**3 * N00_Re / Vphase

print(f"cell file: {cell_path}")
print(f"particles: {N00_Re.size}")
print(f"f_ee min/max/mean: {f_ee.min():.6e} {f_ee.max():.6e} {f_ee.mean():.6e}")

### Theoretical distribution function

Single-cell analysis of the theoretical and numerical distribution functions. This cell evaluates the theoretical occupation `f_analytic` at each particle's position and momentum direction with `occupation_number` from `shadow_casting_analytic_solution`, and compares it with the numerical `f_ee`.


In [ ]:
_solution_dir = Path(
    "/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/notebooks"
)
if str(_solution_dir) not in sys.path:
    sys.path.insert(0, str(_solution_dir))
import shadow_casting_analytic_solution as shadow

# Same emitter source and opacities as shadow_casting_analytic.ipynb.
kappa_e = 0.00000854544806011998  # 1/cm
kappa_a = 0.00012391882339512724  # 1/cm
mu_e = -3.130432908491162  # MeV
T_e = 5.1189  # MeV
E_MeV = 144.74

def fermi_dirac_occupation(energy_MeV, mu_MeV, temperature_MeV):
    exponent = (energy_MeV - mu_MeV) / temperature_MeV
    if exponent >= 700.0:
        return 0.0
    if exponent > 60.0:
        return float(np.exp(-exponent))
    return float(1.0 / (np.exp(exponent) + 1.0))

R_e = float(R)
R_a = float(r_shadowing_sphere_radius)
center_e = np.asarray(r_sphere_center_cm, dtype=np.float64)
center_a = np.asarray(r_shadowing_sphere_center_cm, dtype=np.float64)
separation = center_a - center_e
d_sep = float(np.linalg.norm(separation))
e_sep = separation / d_sep
S_e = fermi_dirac_occupation(E_MeV, mu_e, T_e)

shadow.configure({
    "kappa_a": kappa_a,
    "kappa_e": kappa_e,
    "S_e": S_e,
    "center_e": center_e,
    "center_a": center_a,
    "R_e": R_e,
    "R_a": R_a,
    "e_sep": e_sep,
    "x_external": d_sep * R_e / (R_e - R_a),
    "x_internal": d_sep * R_e / (R_e + R_a),
    "beta_u": float(np.arcsin((R_e - R_a) / d_sep)),
    "beta_p": float(np.arcsin((R_e + R_a) / d_sep)),
    "d": d_sep,
    "dx": float(dx),
    "dy": float(dy),
    "dz": float(dz),
    "ncell": (int(Nx), int(Ny), int(Nz)),
    "phat": np.zeros((1, 3)),
    "vphase_a": np.zeros(1),
    "n_directions": 1,
    "full_f_bytes": 0,
})

directions = np.column_stack((pupx, pupy, pupz))
directions /= np.linalg.norm(directions, axis=1)[:, None]

# occupation_number(x, y, z, n_hat) is that chord formula at one point
# and one direction. Each particle has its own position and momentum.
f_analytic = np.array([
    shadow.occupation_number(x, y, z, n_hat)
    for x, y, z, n_hat in zip(pos_x, pos_y, pos_z, directions)
])

### Comparing the numerical and theoretical distribution functions

Single-cell analysis of the theoretical and numerical distribution functions. This cell prints the source occupation `S_e`, the range of `f_analytic`, the median ratio `f_ee / f_analytic` on directions where the theoretical occupation is positive, and the numerical and theoretical value for every particle in the cell.


In [ ]:
print(f"S_e = {S_e:.6e}")
print(f"f_analytic min/max/mean: {f_analytic.min():.6e} {f_analytic.max():.6e} {f_analytic.mean():.6e}")
positive = f_analytic > 0.0
if np.any(positive):
    ratio = f_ee[positive] / f_analytic[positive]
    print(f"f_ee / f_analytic on {positive.sum()} directions with f_analytic > 0: median {np.median(ratio):.6e}")

for i in range(len(f_ee)):
    print(f"f_ee = {f_ee[i]} f_ee_theory = {f_analytic[i]}")